# 03b: Fraud Detection - Model Tuning with Distributed HPO

This notebook demonstrates:
- Using DataConnector to prepare data for distributed tuning
- Defining hyperparameter search spaces with BayesOpt
- Creating training functions that run on SPCS workers
- Running distributed hyperparameter optimization with the Tuner API
- Extracting best model and metrics from tuning results
- Logging tuned model to the same experiment as baseline
- Registering the best model

Snowflake's ML Runtime uses **Ray** under the hood to distribute hyperparameter trials across your SPCS compute pool.

### What is Ray?
[Ray](https://ray.io) is an open-source distributed computing framework that makes it easy to scale Python workloads. Key benefits:
- **Distributed execution**: Automatically parallelizes trials across multiple nodes
- **Fault tolerance**: Recovers from worker failures without losing progress
- **Resource management**: Efficiently schedules work across available CPUs/GPUs
- **Ray Tune**: Built-in hyperparameter tuning library with search algorithms

### Import Libraries
Import Snowflake ML libraries for distributed tuning, data connectors, registry, and experiment tracking.

In [ ]:
import os
import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model import type_hints
from datetime import datetime

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME")
    )
    session = Session.builder.configs({"connection": conn}).create()

try:
    from snowflake.ml.modeling.tune import Tuner
    HAS_TUNER = True
except ImportError:
    HAS_TUNER = False

print(f"Connected: {session.get_current_account()}")
print(f"Tuner API available: {HAS_TUNER}")

### Set Database Context
Configure the database and schema for this session.

In [ ]:
DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
WAREHOUSE = "ML_WH"
MODEL_NAME = "FRAUD_DETECTION_MODEL"
COMPUTE_POOL = "ML_TRAINING_POOL"
STAGE_NAME = "ML_MODELS"

RUN_REMOTE = not HAS_TUNER

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
print(f"Context: {DATABASE}.{SCHEMA} | Warehouse: {WAREHOUSE}")
print(f"Run Mode: {'@remote (ML Job)' if RUN_REMOTE else 'In-process (Container Runtime)'}")

### Connect to Experiment
Connect to the same experiment used in 03a to track both baseline and tuned models together.

In [ ]:
exp = ExperimentTracking(session=session)
EXPERIMENT_NAME = "FRAUD_DETECTION_EXPERIMENT"
exp.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME} (same as baseline)")

## Load Data via DataConnector

### Load Fraud Training Data
Load `FRAUD_TRAINING_DATA` (same source as 03a) and split into train/test with stratification to preserve fraud rate.

In [ ]:
fraud_df = session.table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")

label_col = "IS_FRAUD"
exclude_cols = ["TRANSACTION_ID", label_col]
feature_cols = [c for c in fraud_df.columns if c not in exclude_cols]

train_df = fraud_df.sample(frac=0.8)
test_df = fraud_df.subtract(train_df)

print(f"Train: {train_df.count()} rows")
print(f"Test:  {test_df.count()} rows")
print(f"Features: {len(feature_cols)}")

### Create DataConnector
Wrap Snowpark DataFrames in `DataConnector` before passing them to the Tuner.

**Why not just use `.to_pandas()` here?**

Calling `.to_pandas()` outside the training function pulls the **entire dataset into the notebook's memory** before distribution. For large datasets (GBs–TBs), this is impractical and slow.

**What DataConnector does instead:**
- Keeps data in Snowflake — nothing is materialized until a worker actually needs it
- Each SPCS worker calls `dm["train"].to_pandas()` independently, pulling only **its assigned shard** 
- Data streams in chunks from Snowflake directly to the worker — no full dataset in memory at any point

**For very large datasets (TBs)**, skip `.to_pandas()` inside `train_func` entirely and use XGBoost's native `DMatrix` instead, which accepts `DataConnector` directly and reads data iteratively without ever loading the full dataset into RAM:

In [ ]:
TRAIN_TABLE = f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA"
NUM_TRIALS = 10
print(f"Training table: {TRAIN_TABLE}")
print(f"Trials: {NUM_TRIALS}")

## Define Hyperparameter Search Space

The Snowflake Tuner API supports three search algorithms:

| Algorithm | Class | Best For |
|-----------|-------|----------|
| **RandomSearch** | `RandomSearch()` | Good baseline; fast exploration of large spaces |
| **GridSearch** | `GridSearch()` | Exhaustive search over discrete values (requires lists, not sampling functions) |
| **BayesOpt** | `BayesOpt()` | Expensive evaluations; uses Gaussian processes to guide search intelligently |

Search space parameters are defined with sampling functions:
- `uniform(lower, upper)` — continuous float
- `loguniform(lower, upper)` — log-scaled float (e.g., learning rates)
- `randint(lower, upper)` — integer (upper exclusive)
- `choice([options])` — categorical (not supported with BayesOpt)

Below we use **BayesOpt** with continuous ranges and UCB acquisition, which is well-suited for tuning XGBoost where each trial is moderately expensive. Includes `scale_pos_weight` to handle class imbalance in fraud detection data.

In [ ]:
SEARCH_SPACE = {
    "n_estimators": (50, 300),
    "max_depth": (3, 10),
    "learning_rate": (0.01, 0.3),
    "scale_pos_weight": (1, 20),
}

print("BayesOpt search space (continuous uniform):")
for k, v in SEARCH_SPACE.items():
    print(f"  {k}: {v}")

## Define Training Function

### Worker Training Function
This function executes on each SPCS worker. It receives hyperparameters from the Tuner, trains an XGBoost model on fraud data, and reports metrics back.

In [ ]:
print("Training function will be defined inside run_hpo()")

## Configure and Run Tuner

### Configure Tuner
Set the optimization metric (AUC), search algorithm, and parallelism settings.

In [ ]:
print(f"Tuner config:")
print(f"  Algorithm: BayesOpt (UCB acquisition)")
print(f"  Metric: auc (maximize)")
print(f"  Trials: {NUM_TRIALS}")
print(f"  Concurrent: 1")

### Ray Dashboard

The Tuner API runs on Ray under the hood. You can monitor active tuning jobs, individual trial progress, cluster resource utilization (CPU/GPU/memory), and per-trial logs via the **Ray Dashboard** — a web UI served by the Ray head node inside your SPCS container. Open the URL in a new tab and log in with your Snowflake credentials.

In [ ]:

from snowflake.ml.runtime_cluster import get_ray_dashboard_url

dashboard_url = get_ray_dashboard_url()
print(f"Ray Dashboard: {dashboard_url}")

### Run Distributed HPO
Launch the Tuner to run parallel trials across SPCS workers. Each trial trains with different hyperparameters.

In [ ]:
def run_hpo(session, train_table, search_space_bounds, num_trials):
    """Run HPO - works both in Container Runtime and via @remote"""
    from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context
    from snowflake.ml.modeling.tune.search import BayesOpt
    from snowflake.ml.modeling import tune
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.ml.registry import Registry
    from snowflake.ml.model import type_hints
    import pickle
    import os
    
    # Load full table and split into 80/20 train/test
    print("Loading training data...")
    fraud_df = session.table(train_table)
    train_df = fraud_df.sample(frac=0.8)
    test_df = fraud_df.subtract(train_df)
    print(f"Train: {train_df.count()}, Test: {test_df.count()}")
    
    # Wrap Snowpark DataFrames in DataConnectors for the Tuner API
    dataset_map = {
        "train": DataConnector.from_dataframe(train_df),
        "test": DataConnector.from_dataframe(test_df),
    }
    
    # Convert bounds dict to continuous uniform distributions for each hyperparam
    search_space = {k: tune.uniform(v[0], v[1]) for k, v in search_space_bounds.items()}
    
    # Configure Tuner: maximize AUC using Bayesian optimization (UCB acquisition)
    tuner_config = TunerConfig(
        metric="auc",
        mode="max",
        search_alg=BayesOpt(utility_kwargs={"kind": "ucb", "kappa": 2.5}),
        num_trials=num_trials,
        max_concurrent_trials=1,
    )
    
    # train_func runs on each Ray worker per trial — must be self-contained
    # (Ray pickles this function and ships it to workers, so all imports go inside)
    def train_func():
        from xgboost import XGBClassifier
        from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
        
        # Get this trial's context: sampled hyperparams and the dataset map
        tuner_context = get_tuner_context()
        config = tuner_context.get_hyper_params()
        dm = tuner_context.get_dataset_map()
        
        # Convert DataConnectors to pandas for XGBoost
        train_pd = dm["train"].to_pandas()
        test_pd = dm["test"].to_pandas()
        
        # Derive feature columns by excluding IDs and the label
        label_col = "IS_FRAUD"
        exclude_cols = ["TRANSACTION_ID", "CUSTOMER_ID", label_col]
        feature_cols = [c for c in train_pd.columns if c not in exclude_cols]
        X_train, y_train = train_pd[feature_cols], train_pd[label_col]
        X_test, y_test = test_pd[feature_cols], test_pd[label_col]
        
        # Build XGBClassifier with this trial's sampled hyperparams
        model = XGBClassifier(
            n_estimators=int(config["n_estimators"]),
            max_depth=int(config["max_depth"]),
            learning_rate=config["learning_rate"],
            scale_pos_weight=config["scale_pos_weight"],
            random_state=42,
            eval_metric="logloss"
        )
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        # Report metrics back to the Tuner; "auc" must match TunerConfig.metric
        tuner_context.report(
            metrics={
                "accuracy": accuracy_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred),
                "auc": roc_auc_score(y_test, y_proba),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
            },
            model=model
        )
    
    # Launch distributed HPO — Tuner orchestrates Ray trials on SPCS
    print("Starting distributed HPO...")
    tuner = Tuner(train_func, search_space, tuner_config)
    results = tuner.run(dataset_map=dataset_map)
    print("HPO complete!")
    
    # Rebuild feature list from the full table (for model registration sample_input)
    label_col = "IS_FRAUD"
    exclude_cols = ["TRANSACTION_ID", "CUSTOMER_ID", label_col]
    feature_cols = [c for c in fraud_df.columns if c not in exclude_cols]
    
    # Extract the winning trial's hyperparams
    best_params = {
        "n_estimators": int(results.best_result["config/n_estimators"].values[0]),
        "max_depth": int(results.best_result["config/max_depth"].values[0]),
        "learning_rate": float(results.best_result["config/learning_rate"].values[0]),
        "scale_pos_weight": float(results.best_result["config/scale_pos_weight"].values[0]),
    }
    # Extract the winning trial's evaluation metrics
    best_metrics = {
        "accuracy": float(results.best_result["accuracy"].values[0]),
        "f1": float(results.best_result["f1"].values[0]),
        "auc": float(results.best_result["auc"].values[0]),
        "precision": float(results.best_result["precision"].values[0]),
        "recall": float(results.best_result["recall"].values[0]),
    }
    
    # Parse DB/schema from the fully qualified table name for registry
    db_parts = train_table.split(".")
    DATABASE, SCHEMA = db_parts[0], db_parts[1]
    
    # Register the best model to the Snowflake Model Registry
    reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
    from datetime import datetime
    version_name = f"TUNED_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Provide a sample input for schema inference during registration
    sample_df = session.table(train_table).select(list(feature_cols)).na.fill(0).limit(100)
    
    mv = reg.log_model(
        model=results.best_model,
        model_name="FRAUD_DETECTION_MODEL",
        version_name=version_name,
        sample_input_data=sample_df,
        comment="XGBoost fraud detection - tuned via HPO",
        metrics=best_metrics,
        target_platforms=[type_hints.TargetPlatform.SNOWPARK_CONTAINER_SERVICES]
    )
    print(f"Registered model: FRAUD_DETECTION_MODEL:{version_name}")
    
    return {
        "best_params": best_params,
        "best_metrics": best_metrics,
        "feature_cols": list(feature_cols),
        "model_version": version_name,
    }

# Choose execution path: @remote (ML Job on SPCS) or in-process (Container Runtime)
if RUN_REMOTE:
    from snowflake.ml.jobs import remote
    
    # Wrap run_hpo in @remote to ship it to a compute pool as an ML Job
    @remote(compute_pool=COMPUTE_POOL, stage_name=STAGE_NAME, session=session)
    def run_hpo_remote(train_table, search_space_bounds, num_trials):
        from snowflake.snowpark import Session
        sess = Session.builder.getOrCreate()  # create server-side session on SPCS
        return run_hpo(sess, train_table, search_space_bounds, num_trials)
    
    print("Submitting HPO job to SPCS via @remote...")
    hpo_job = run_hpo_remote(TRAIN_TABLE, SEARCH_SPACE, NUM_TRIALS)
    print(f"Job ID: {hpo_job.id}")
    print(f"Status: {hpo_job.status}")
    print("\nWaiting for job to complete (10+ minutes)...")
    hpo_job.wait()
    print(f"Final status: {hpo_job.status}")
    
    if hpo_job.status == "DONE":
        hpo_output = hpo_job.result()
        best_params = hpo_output["best_params"]
        best_metrics = hpo_output["best_metrics"]
        feature_cols = hpo_output["feature_cols"]
        model_version = hpo_output["model_version"]
        print(f"\nModel registered: FRAUD_DETECTION_MODEL:{model_version}")
        print(f"Best AUC: {best_metrics['auc']:.4f}")
        print(f"Best params: {best_params}")
        # Load the registered model back for downstream use
        best_model = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA).get_model("FRAUD_DETECTION_MODEL").version(model_version).load(force=True)
    else:
        print("\n=== Logs ===")
        print(hpo_job.get_logs())
        raise RuntimeError("HPO job failed")
else:
    # Run HPO directly in the notebook's Container Runtime (no job submission)
    hpo_output = run_hpo(session, TRAIN_TABLE, SEARCH_SPACE, NUM_TRIALS)
    best_params = hpo_output["best_params"]
    best_metrics = hpo_output["best_metrics"]
    feature_cols = hpo_output["feature_cols"]
    model_version = hpo_output["model_version"]
    print(f"\nBest AUC: {best_metrics['auc']:.4f}")
    print(f"Best params: {best_params}")
    # Load the registered model back for downstream use
    best_model = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA).get_model("FRAUD_DETECTION_MODEL").version(model_version).load(force=True)

## View Results

### Extract Best Model and Metrics
Retrieve the best performing model and its hyperparameters from the tuning results.

In [ ]:
print(f"Best params: {best_params}")
print(f"Best metrics: {best_metrics}")
print(f"Model version: {model_version}")

## Log Tuned Model to Experiment

### Log Best Run to Experiment
Record the best hyperparameters and metrics to the same `FRAUD_DETECTION_EXPERIMENT` as the baseline model for comparison.

In [ ]:
TUNED_RUN_NAME = f"xgboost_tuned_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with exp.start_run(TUNED_RUN_NAME):
    exp.log_params({
        "model_type": "XGBClassifier",
        "compute": "SPCS",
        "tuning_method": "BayesOpt",
        "num_trials": NUM_TRIALS,
        **best_params
    })
    exp.log_metrics(best_metrics)

print(f"Logged tuned run: {TUNED_RUN_NAME}")

## Register Tuned Model

### Verify Model Registration
The tuned model was already registered inside `run_hpo()` with a timestamped version name. Verify it exists in the registry.

In [ ]:
reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

model_ref = reg.get_model(MODEL_NAME)
versions = model_ref.show_versions()
print(f"Model: {MODEL_NAME}")
print(f"Tuned version: {model_version}")
print(f"\nAll versions:")
for _, row in versions.iterrows():
    print(f"  - {row['name']}")
print(f"\nBest AUC: {best_metrics['auc']:.4f}")

### Verify Registration
Query the Model Registry to confirm both model versions are registered.

In [ ]:
%%sql -r dataframe_1
SHOW VERSIONS IN MODEL DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.FRAUD_DETECTION_MODEL;